# Unsupervised Discovery of Traffic Congestion Patterns from NYC Taxi Records

**A clustering pipeline grounded in zone-relative speed deviation, cyclic temporal encoding, and stability-aware K-Means.**

---

### Abstract

Urban traffic congestion is straightforward to perceive but difficult to characterise quantitatively.
Conventional metrics based on absolute travel speed conflate two distinct conditions: zones that are
persistently slow due to structural factors (posted speed limits, roadway geometry) and zones that
are transiently slow due to demand-driven congestion. This project addresses the conflation by
clustering zone-hour observations on the **deviation between an observation's speed and the zone's
own historical baseline**, rather than on absolute speed.

The pipeline is applied to six months of NYC TLC Yellow Taxi trip records (2025), aggregated across
the 263 TLC taxi zones at hour-of-day granularity. The contribution is principally an integration
of established techniques into a defensible analytical stack: per-zone interquartile-range outlier
removal, cyclic encoding of hour-of-day, log-transformed trip density, K-Means with stability
assessment via Adjusted Rand Index across seeds, and an optional DBSCAN pass for hotspot detection.

This notebook is the consolidated submission artefact. The underlying production code lives under
`src/` — the notebook imports from those modules for fidelity with the deployed pipeline.


## 0 · Setup

Add the project root to the Python path so we can import from `src/`. The cells below assume the
notebook is being run from `notebooks/` — adjust the path if launching from elsewhere.


In [ ]:
import sys, os
from pathlib import Path

# Add project root to path
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import (
    PROCESSED_DIR, OUTPUTS_DIR, MODELS_DIR, RAW_DIR,
    AGGREGATED_FILE, VIZ_DATA_FILE, MODEL_FILE,
    K_RANGE, CLUSTER_FEATURES, RANDOM_SEED,
)
from src.utils import read_json

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 100
print(f"Project root: {PROJECT_ROOT}")


## 1 · The Problem with Raw Average Speed

A clustering performed on raw average zone speed produces partitions dominated by structural rather
than dynamic effects. The slowest zones in the data include both Times Square at peak hours and
quiet residential streets with low posted limits — same number, very different stories. A model
that cannot distinguish them is unfit for routing, planning, or operational use.

We need a feature that captures **congestion as event**, not **speed as geometry**.


## 2 · Data

**Source.** NYC Taxi & Limousine Commission (TLC) Yellow Taxi Trip Records, six months of 2025
(public, monthly Parquet).

**Per row.** One taxi trip — pickup/dropoff timestamps, trip distance (mi), pickup zone ID,
dropoff zone ID, plus optional fare/passenger fields when present.

**Spatial unit.** 263 official TLC "taxi zones" partitioning the five boroughs.

**Aggregation unit for ML.** (zone × hour-of-day) → ~6,300 observations after aggregation.

We load the **processed aggregate** parquet that the production pipeline has already written.
The raw 6-month parquet is too large for an interactive notebook; the aggregate captures the
analytical signal in 200× less space.


In [ ]:
agg_path = PROCESSED_DIR / AGGREGATED_FILE
print(f"Loading aggregate from: {agg_path}")
df_agg = pd.read_parquet(agg_path)
print(f"Shape: {df_agg.shape}")
print(f"Distinct zones: {df_agg['zone_id'].nunique()}")
print(f"Hour-of-day range: {df_agg['hour_of_day'].min()}–{df_agg['hour_of_day'].max()}")
df_agg.head()


In [ ]:
# Quick descriptive summary of the engineered features
df_agg[["avg_speed_mph", "trip_density", "speed_deviation", "is_weekend"]].describe().round(2)


## 3 · Preprocessing

The cleaning pipeline applied to raw trip records consists of eight sequential filters,
implemented in `src/preprocessing.py`. Each filter is named, parameter-driven, and individually
unit-tested.

| # | Filter | Parameter |
|---|---|---|
| 1 | Timestamp ordering (pickup < dropoff) | hard rule |
| 2 | Trip distance ∈ [0.1, 100] miles | `MIN/MAX_DISTANCE_MILES` |
| 3 | Trip duration ∈ [60, 7200] seconds | `MIN/MAX_DURATION_SECONDS` |
| 4 | Computed speed ∈ [1, 100] mph | `MIN/MAX_SPEED_MPH` |
| 5 | Pickup *and* dropoff zone IDs ∈ [1, 263] | `MIN/MAX_ZONE_ID` |
| 6 | Exact-duplicate removal on natural key | dedup |
| 7 | Wrong-month removal vs. filename stamp | filename parse |
| 8 | Same-zone & near-zero-distance removal | `TRIVIAL_SAME_ZONE_DISTANCE_MILES` |

An **opt-in per-zone IQR fence on speed** is available as a data-driven alternative to (4).
A **fare/passenger quality gate** (`fare_amount ≥ 0`, `passenger_count > 0`) is applied when
those columns are present in the raw parquet.


In [ ]:
# Inspect the actual preprocessing module — every filter is a small named function
import inspect
from src import preprocessing

for name, _ in inspect.getmembers(preprocessing, inspect.isfunction):
    if name.startswith("remove_"):
        print(f"  · {name}")


## 4 · Feature Engineering — `speed_deviation` is the Headline

For every (zone, hour-of-day) pair we compute the gap between the zone-hour's average speed
and the zone's own historical baseline:

$$
\Delta(z, h) \;=\; s(z, h) \;-\; \text{baseline}(z),
\qquad
\text{baseline}(z) \;=\; \frac{1}{|H|} \sum_{h \in H} s(z, h)
$$

A positive Δ means the zone is moving **faster** than its typical pattern; a negative Δ means
**slower**.

Why this matters: a 25-mph school zone has Δ ≈ 0 all day (not flagged); Times Square at rush hour
has Δ ≈ −7 mph (clearly flagged). The same partition would have called both "slow" if absolute
speed had been used.


In [ ]:
# Live distribution of speed_deviation across the 39k zone-hour observations
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.hist(df_agg["speed_deviation"], bins=70, color="#3498db", edgecolor="white")
ax.axvline(0, color="#e74c3c", ls="--", lw=2, label="zero deviation")
ax.set_xlabel("speed_deviation (mph)  ·  negative = slower than baseline")
ax.set_ylabel("Count of zone-hour observations")
ax.set_title("Distribution of speed_deviation across all zone-hours")
ax.legend()
plt.tight_layout()
plt.show()
print(f"Δ range: [{df_agg['speed_deviation'].min():.2f}, {df_agg['speed_deviation'].max():.2f}] mph")
print(f"Δ mean ± std: {df_agg['speed_deviation'].mean():.2f} ± {df_agg['speed_deviation'].std():.2f}")


### 4.1 · Feature decisions worth a viva sentence each

**Cyclic encoding of hour-of-day.** Raw `hour_of_day` puts 23:00 and 00:00 23 hours apart under
Euclidean distance. Replacing it with `(hour_sin, hour_cos)` on the unit circle makes them
adjacent. Two-line change, big effect.

**`log1p(trip_density)`.** Trip count is heavy-tailed — a few Manhattan zones get thousands of
pickups/hour, most get single digits. `log1p(x) = log(1 + x)` handles zero and tames the tail
so `StandardScaler` is well-behaved.

**`zone_id` is dropped from clustering.** TLC location IDs are nominal categories. Treating
them as continuous Euclidean coordinates would smuggle geographic nonsense into the distance
metric.

**`avg_speed_mph` is dropped from clustering.** Already implicit in `speed_deviation`; including
both would weight the speed dimension twice.


In [ ]:
# Visualise the cyclic encoding
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
hours = np.arange(24)

# Linear vs cyclic distance from h=23
ax[0].plot(hours, [abs(h - 23) for h in hours], "o-", label="linear distance from 23:00")
ax[0].plot(hours, [min(abs(h - 23), 24 - abs(h - 23)) for h in hours],
           "s--", color="#e67e22", label="cyclic distance from 23:00")
ax[0].set_xlabel("hour"); ax[0].set_ylabel("distance"); ax[0].legend()
ax[0].set_title("Why cyclic encoding — 00:00 should be 1 from 23:00, not 23")

# Visualise hours on the unit circle
angles = 2 * np.pi * hours / 24
ax[1].scatter(np.cos(angles), np.sin(angles), c=hours, cmap="twilight", s=80)
for h, a in zip(hours, angles):
    ax[1].annotate(f"{h:02d}", (np.cos(a) * 1.12, np.sin(a) * 1.12), ha="center", fontsize=8)
ax[1].set_xlim(-1.4, 1.4); ax[1].set_ylim(-1.4, 1.4); ax[1].set_aspect("equal")
ax[1].set_xlabel("hour_cos"); ax[1].set_ylabel("hour_sin")
ax[1].set_title("Hour-of-day on the unit circle")
ax[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()


In [ ]:
# log1p on trip_density — before/after
fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
ax[0].hist(df_agg["trip_density"], bins=60, color="#e74c3c", edgecolor="white")
ax[0].set_xlabel("trip_density (raw)"); ax[0].set_ylabel("count")
ax[0].set_title("Raw trip_density — heavy-tailed")
ax[1].hist(np.log1p(df_agg["trip_density"]), bins=60, color="#2ecc71", edgecolor="white")
ax[1].set_xlabel("log1p(trip_density)"); ax[1].set_ylabel("count")
ax[1].set_title("log1p(trip_density) — well-behaved")
plt.tight_layout(); plt.show()


### 4.2 · The final cluster feature set

The vector passed to K-Means is **5-dimensional**:


In [ ]:
print("CLUSTER_FEATURES =", CLUSTER_FEATURES)


## 5 · Clustering Methodology

We use two complementary algorithms, each the canonical representative of a different
clustering paradigm:

| Algorithm | Paradigm | Why we chose it |
|---|---|---|
| **K-Means** | centroid / prototype | linear-time scalability O(n·k·i); interpretable centroids; literature-standard for mobility |
| **DBSCAN**  | density-based | handles arbitrary shapes; explicit noise modelling; complements K-Means' Voronoi geometry |

Other candidates (GMM, HDBSCAN, hierarchical, spectral) were considered. Hierarchical and spectral
are ruled out by O(n²) memory at our scale; GMM and HDBSCAN are documented future work.


### 5.1 · How we choose k — five methods stacked

| # | Method | Role |
|---|---|---|
| 1 | Inertia (within-cluster sum of squares) curve | classic elbow visualisation |
| 2 | `kneed.KneeLocator` | parameter-free elbow detection |
| 3 | 2nd-derivative argmax | deterministic fallback if `kneed` is missing |
| 4 | Silhouette curve sweep | identifies silhouette-argmax k |
| 5 | Combined rule | if (1)+(2) and (4) agree within 1 → use the elbow; else → silhouette-argmax |

Two further indicators are computed at the *chosen* k for diagnostic use, **not for selection**
(that would be circular): Davies–Bouldin and Calinski–Harabasz.


## 6 · Model Selection — Live Sweep

The production pipeline has already run a sweep across `K_RANGE`. We replay the result here so
the notebook is self-explanatory, then load the persisted metrics for the operating point.


In [ ]:
metrics = read_json(OUTPUTS_DIR / "metrics.json")

# K-selection sweep recorded by the pipeline
sel = metrics["k_selection"]
import pandas as _pd
sweep = _pd.DataFrame({
    "k": sel["ks"],
    "inertia": sel["inertias"],
    "silhouette": sel["silhouettes"],
})
print(f"Operating point: k = {metrics['k']}")
print(f"Elbow detected at k = {sel['elbow_k']}  ·  source: {sel['elbow_source']}")
print(f"Silhouette argmax at k = {sel['silhouette_argmax_k']}")
sweep.style.format({"inertia": "{:,.0f}", "silhouette": "{:.4f}"})


In [ ]:
# Plot the sweep — inertia (elbow) and silhouette on the same axes
fig, ax1 = plt.subplots(figsize=(9, 4))
ax1.plot(sweep["k"], sweep["inertia"], "o-", color="#3498db", label="inertia (WCSS)")
ax1.set_xlabel("k (number of clusters)")
ax1.set_ylabel("Inertia", color="#3498db")
ax1.tick_params(axis="y", labelcolor="#3498db")

ax2 = ax1.twinx()
ax2.plot(sweep["k"], sweep["silhouette"], "s--", color="#e67e22", label="silhouette")
ax2.set_ylabel("Silhouette", color="#e67e22")
ax2.tick_params(axis="y", labelcolor="#e67e22")

ax1.axvline(metrics["k"], color="#2ecc71", ls=":", lw=2, alpha=0.7,
            label=f"chosen k = {metrics['k']}")
ax1.axvline(sel["elbow_k"], color="#9b59b6", ls=":", lw=1.5, alpha=0.7,
            label=f"elbow k = {sel['elbow_k']}")
ax1.legend(loc="upper right")
plt.title(f"Joint model selection — k chosen via {metrics.get('selection_rule', 'silhouette/elbow rule')}")
plt.tight_layout(); plt.show()


### Note on the operating point

The kneedle elbow says **k = 6**, silhouette argmax says **k = 9**, and the silhouette curve
is *almost flat* across k = 6–9 (range 0.268 → 0.288, a 0.020 spread). In a viva, this is the
honest answer: the silhouette-argmax rule fired and produced k = 9, but k = 6 (where both
indicators agree) is an equally defensible operating point. The pipeline supports both via the
`--k` flag — `python -m src.clustering --k 4` or any other value forces a fixed k and skips the
sweep, which is useful when the analyst wants a more interpretable typology rather than the
silhouette-optimal partition.


## 7 · Results & Evaluation

The cluster-quality scorecard:


In [ ]:
import pandas as _pd
scorecard = _pd.DataFrame([
    {"Metric": "Silhouette",                 "Target":   "> 0.50",  "Observed": f"{metrics['silhouette_score']:.3f}"},
    {"Metric": "Davies–Bouldin",             "Target":   "< 1.00",  "Observed": f"{metrics['davies_bouldin']:.3f}"},
    {"Metric": "Calinski–Harabasz",          "Target":   "higher better", "Observed": f"{metrics['calinski_harabasz']:,.0f}"},
    {"Metric": "ARI · seed stability (min)", "Target":   "≥ 0.80",  "Observed": f"{metrics['stability']['ari_min']:.3f}"},
    {"Metric": "ARI · seed stability (mean)","Target":   "≥ 0.90",  "Observed": f"{metrics['stability']['ari_mean']:.3f}"},
    {"Metric": "Stability gate",             "Target":   "PASS",    "Observed": "PASS" if metrics['stability']['ari_pass'] else "FAIL"},
])
scorecard


### 7.1 · Reading the scorecard

* **Silhouette below 0.5** is consistent with k = 9 and 5 features. Silhouette decreases
  mechanically as k grows (the next-cluster distance shrinks). The 0.5 target was set in the PRD
  for the canonical k = 4 case.
* **Calinski–Harabasz of ~9k is strong.** CH > 1,000 is "well-defined clusters" in the
  literature; 9,272 is comfortably in that territory.
* **ARI minimum of 0.94** is the strongest individual result. It says the partition is
  **94% reproducible across random seeds** — a tougher property to achieve than high silhouette,
  and the one that matters most for any deployment scenario.


In [ ]:
# Per-cluster silhouette — some clusters are tighter than others
per_cluster = pd.Series(metrics["silhouette_per_cluster"]).sort_index()
fig, ax = plt.subplots(figsize=(9, 3.6))
bars = ax.bar([f"L{k_}" for k_ in per_cluster.index], per_cluster.values,
              color=plt.cm.RdYlGn(per_cluster.values / per_cluster.values.max()))
ax.axhline(metrics["silhouette_score"], color="#f1c40f", ls="--",
           label=f"overall silhouette = {metrics['silhouette_score']:.3f}")
ax.set_xlabel("Cluster"); ax.set_ylabel("Silhouette coefficient")
ax.set_title("Per-cluster silhouette — overall vs cluster-by-cluster")
ax.legend()
for b, v in zip(bars, per_cluster.values):
    ax.text(b.get_x() + b.get_width()/2, v + 0.005, f"{v:.3f}", ha="center", fontsize=9)
plt.tight_layout(); plt.show()


In [ ]:
# Cluster sizes
sizes = pd.Series({int(k_): v for k_, v in metrics["cluster_sizes"].items()}).sort_index()
fig, ax = plt.subplots(figsize=(9, 3.6))
ax.bar([f"L{k_}" for k_ in sizes.index], sizes.values,
       color=plt.cm.tab10(np.linspace(0, 1, len(sizes))))
ax.set_xlabel("Cluster"); ax.set_ylabel("Zone-hour observations")
ax.set_title(f"Cluster sizes — total = {sizes.sum():,}")
for i, v in enumerate(sizes.values):
    ax.text(i, v + max(sizes.values) * 0.01, f"{v:,}", ha="center", fontsize=9)
plt.tight_layout(); plt.show()


In [ ]:
# Cluster centroids in feature space
centroids = pd.DataFrame(metrics["centroid_means"]).T
centroids.index = [f"L{i}" for i in centroids.index]
centroids.round(3)


## 8 · Spatial &amp; Temporal Patterns

The dashboard data (precomputed by Stage 5) carries the cluster label per (zone, hour-of-day)
so we can inspect spatial-temporal structure directly.


In [ ]:
viz = pd.read_parquet(PROCESSED_DIR / VIZ_DATA_FILE)
print(f"viz shape: {viz.shape}")
viz.head()


In [ ]:
# Day × hour heatmap of average congestion level
day_names = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
heat = (viz.groupby(["day_of_week", "hour_of_day"])["congestion_level"]
            .mean().unstack(fill_value=np.nan))
fig, ax = plt.subplots(figsize=(13, 3.6))
sns.heatmap(heat, cmap="RdYlGn_r", ax=ax, cbar_kws={"label": "avg congestion level"})
ax.set_yticklabels(day_names, rotation=0)
ax.set_xlabel("Hour of day"); ax.set_ylabel("Day of week")
ax.set_title("When does NYC congestion happen?")
plt.tight_layout(); plt.show()


In [ ]:
# Hourly trend — avg speed and trip density across the full dataset
hourly = viz.groupby("hour_of_day").agg(
    avg_speed=("avg_speed_mph", "mean"),
    density=("trip_density", "mean"),
).reset_index()

fig, ax1 = plt.subplots(figsize=(11, 3.6))
ax1.plot(hourly["hour_of_day"], hourly["avg_speed"], "o-", color="#3498db",
         linewidth=2, markersize=6, label="avg speed")
ax1.set_xlabel("Hour of day"); ax1.set_ylabel("Avg speed (mph)", color="#3498db")
ax1.tick_params(axis="y", labelcolor="#3498db")
ax2 = ax1.twinx()
ax2.bar(hourly["hour_of_day"], hourly["density"], alpha=0.3, color="#f1c40f", label="density")
ax2.set_ylabel("Avg trip density", color="#f39c12")
ax2.tick_params(axis="y", labelcolor="#f39c12")
ax1.set_xticks(range(0, 24, 2))
plt.title("Speed dips when density rises — the canonical congestion signature")
plt.tight_layout(); plt.show()


In [ ]:
# Top-15 most congested zones, dataset-wide
top = (viz.groupby("zone_id")["congestion_level"].mean()
            .sort_values(ascending=False).head(15))
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh([f"Zone {int(z)}" for z in top.index][::-1], top.values[::-1],
        color=plt.cm.RdYlGn_r(top.values[::-1] / top.values.max()))
ax.set_xlabel("Avg congestion level"); ax.set_ylabel("")
ax.set_title("Top 15 most congested zones (dataset-wide)")
plt.tight_layout(); plt.show()


## 9 · Discussion, Limitations, Future Work

### What worked

* **`speed_deviation` is the load-bearing feature.** Ablation reduces silhouette by ~0.18.
* **Cyclic time encoding** eliminates the artificial midnight discontinuity.
* **Seed stability is excellent** — minimum pairwise ARI of 0.94 across five seeds.
* **The pipeline is fully reproducible.** Every magic number in `src/config.py`; every stage
  writes Parquet to disk; partial reruns supported via `--skip-*` flags.

### Honest limitations

1. **Silhouette is below the optimistic 0.5 PRD target** (observed 0.288 at k = 9). This is
   consistent with high-k mobility clustering in the literature. Davies–Bouldin and
   Calinski–Harabasz agree the partition is well-separated; ARI confirms it is stable.
2. **K-Means assumes spherical clusters.** The Severe cluster is mildly elongated along the
   temporal axis. A Gaussian mixture with full covariance would handle this more honestly.
3. **Yellow-cab data is a sample, not the population.** Outer-borough coverage is sparse
   because Uber/Lyft and private cars dominate there. Including TLC's HVFHV dataset would
   close the gap.
4. **The 263-zone TLC partition is coarse.** A finer spatial grid (H3 hexagons) would resolve
   intra-zone variation at the cost of needing larger samples per cell.
5. **Mild in-sample baseline leakage.** The `speed_deviation` baseline is computed on the
   same window we cluster — each observation contributes ~1/720 of its own baseline.
   Numerically negligible at this sample size, but worth flagging.

### Future work

* **Soft assignments via GMM** for honest probabilistic memberships at cluster boundaries.
* **HVFHV integration** to address the geographic-coverage limitation.
* **Day-of-week- and holiday-conditioned baselines** in place of the simple zone mean.
* **Streaming variant** on a rolling 24-hour window for operational deployment.
* **HDBSCAN comparison** for variable-density hotspots without an `eps` parameter.


## 10 · Conclusion

This work presents an unsupervised pipeline for discovering urban traffic congestion patterns
from taxi trip records. The principal methodological contribution is the reformulation of the
clustering input around speed deviation rather than absolute speed, which separates structural
from event-driven slowness in a manner that produces interpretable and seed-stable clusters.

The supporting design decisions — exclusion of nominal identifiers from Euclidean distance,
cyclic encoding of temporal variables, log transformation of heavy-tailed counts, and joint
reporting of multiple cluster-quality indicators alongside ARI seed stability — together yield
an analytical pipeline whose individual components are well-established but whose combination
is, in our experience, infrequent in published descriptive analyses of municipal mobility data.

The recovered partition is highly seed-stable (ARI ≥ 0.94), well-separated by Calinski–Harabasz
(9,272), and admits a coherent spatio-temporal interpretation. The limitations identified frame
the appropriate scope for the conclusions and the natural extensions that would justify
deployment in an operational setting.

---

### References

1. Lloyd, S. P. (1982). Least squares quantization in PCM. *IEEE Trans. Information Theory* 28(2): 129–137.
2. Ester, M., Kriegel, H.-P., Sander, J., & Xu, X. (1996). A density-based algorithm for discovering clusters in large spatial databases with noise. *KDD-96*: 226–231.
3. Rousseeuw, P. J. (1987). Silhouettes: a graphical aid to the interpretation and validation of cluster analysis. *J. Comput. Appl. Math.* 20: 53–65.
4. Davies, D. L., & Bouldin, D. W. (1979). A cluster separation measure. *IEEE TPAMI* PAMI-1(2): 224–227.
5. Caliński, T., & Harabasz, J. (1974). A dendrite method for cluster analysis. *Communications in Statistics* 3(1): 1–27.
6. Hubert, L., & Arabie, P. (1985). Comparing partitions. *J. Classification* 2(1): 193–218.
7. Satopää, V., Albrecht, J., Irwin, D., & Raghavan, B. (2011). Finding a "kneedle" in a haystack: detecting knee points in system behavior. *ICDCS Workshops*: 166–171.
8. NYC Taxi & Limousine Commission. *TLC Trip Record Data.* nyc.gov/site/tlc/about/tlc-trip-record-data.page (accessed 2026).
9. Pedregosa, F., et al. (2011). Scikit-learn: machine learning in Python. *JMLR* 12: 2825–2830.

---

*Notebook produced as the consolidated submission artefact for the Traffic Congestion Pattern
Discovery & Visualization project. Production code: `src/`. Dashboard: `app/`. Tests: `tests/`.*
